## Create sequential Models based on neural network Using pytorch

In [1]:
import contextlib
import math
import time
from typing import Dict, List, Tuple, Any, Optional
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchinfo import summary
# Import PEFT
from peft import LoraConfig, get_peft_model,TaskType
import loralib as lora
from PIL import Image


# gradio_run_models.py
import gradio as gr
import io
import traceback

import tempfile
import mlflow
from pathlib import Path
from mlflow.tracking import MlflowClient

#### TaskType Can be
* `TaskType.SEQ_CLS` → sequence classification
* `TaskType.TOKEN_CLS` → token classification
* `TaskType.CAUSAL_LM` → causal language modeling
* `TaskType.SEQ_2_SEQ_LM` → seq2seq language modeling
* `TaskType.QUESTION_ANS` → question answering
* `TaskType.FEATURE_EXTRACTION` → feature extraction


In [2]:
# Flag to control normalization
apply_normalize = True   # set to False if you don’t want normalization

# 1. Define preprocessing transforms
if apply_normalize:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean & std
    ])
else:
    transform = transforms.Compose([
        transforms.ToTensor()  # just scale [0,255] → [0,1]
    ])

# 2. Load MNIST dataset
train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# 3. Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 4. Inspect shapes
images, labels = next(iter(train_loader))
print(f"X_train batch shape: {images.shape}")   # (batch_size, 1, 28, 28)
print(f"y_train batch shape: {labels.shape}")   # (batch_size,)

X_train batch shape: torch.Size([64, 1, 28, 28])
y_train batch shape: torch.Size([64])


In [3]:
# Allowed options
_KERNEL_INITS = {"GlorotNormal", "GlorotUniform", "HeNormal", "HeUniform"}
_HIDDEN_CHOICES = {1000, 750, 500, 300, 150, 75, 25}

def _apply_kernel_init(tensor: torch.Tensor, init_name: str):
    if init_name == "GlorotNormal":
        nn.init.xavier_normal_(tensor)
    elif init_name == "GlorotUniform":
        nn.init.xavier_uniform_(tensor)
    elif init_name == "HeNormal":
        nn.init.kaiming_normal_(tensor, nonlinearity="relu")
    elif init_name == "HeUniform":
        nn.init.kaiming_uniform_(tensor, nonlinearity="relu")
    else:
        raise ValueError(f"Unknown kernel init: {init_name}")

class PytorchTensorDigitMist(nn.Module):
    """
    Dense MLP for 28x28 inputs. Hidden size is configurable.
    Output size fixed to 10 (classification).
    """
    def __init__(self, hidden: int = 300, use_bias: bool = True, kernel_init: str = "GlorotUniform"):
        super().__init__()
        if hidden not in _HIDDEN_CHOICES:
            raise ValueError(f"hidden must be one of {_HIDDEN_CHOICES}, got {hidden}")
        if kernel_init not in _KERNEL_INITS:
            raise ValueError(f"kernel_init must be one of {_KERNEL_INITS}, got {kernel_init}")

        self.hidden = hidden
        self.use_bias = use_bias
        self.kernel_init = kernel_init

        self.flatten = nn.Flatten()
        # explicit two-layer topology: hidden -> hidden//2 -> 10
        self.fc1 = nn.Linear(28 * 28, hidden, bias=use_bias)
        self.fc2 = nn.Linear(hidden, hidden // 2, bias=use_bias)
        self.fc3 = nn.Linear(hidden // 2, 10, bias=use_bias)

        self._init_weights()

    def _init_weights(self):
        for layer in (self.fc1, self.fc2, self.fc3):
            if hasattr(layer, "weight") and layer.weight is not None:
                _apply_kernel_init(layer.weight, self.kernel_init)
            if hasattr(layer, "bias") and layer.bias is not None:
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # return raw logits (preferred for CrossEntropyLoss)
        return self.fc3(x)

class PytorchTensorDigitMistLoRA(PytorchTensorDigitMist):
    def __init__(self, hidden=300, use_bias=True, kernel_init="GlorotUniform", r=8):
        super().__init__(hidden, use_bias, kernel_init)
        # Replace dense layers with LoRA-enabled versions
        self.fc1 = lora.Linear(28*28, hidden, r=r, bias=use_bias)
        self.fc2 = lora.Linear(hidden, hidden//2, r=r, bias=use_bias)
        self.fc3 = lora.Linear(hidden//2, 10, r=r, bias=use_bias)


def TensorModels(
    KERNEL_INIT: str = "GlorotUniform",
    hidden: int = 300,
    use_bias: bool = True,
    print_use_config: bool = False,
    lora_uses: bool = False,
    lora_rank: int = 8
) -> Tuple[PytorchTensorDigitMist, Dict[str, object]]:
    """
    Factory returning (model, config). If print_use_config is True, prints the config.
    Args:
      KERNEL_INIT: one of ["GlorotNormal","GlorotUniform","HeNormal","HeUniform"]
      hidden: one of [1000,750,500,300,150,75,25]
      use_bias: whether to include bias terms in Linear layers
      print_use_config: if True, prints the chosen configuration to stdout
    Returns:
      (model, config_dict)
    """
    # normalize common naming variants
    name_map = {
        "glorotnormal": "GlorotNormal",
        "glorotuniform": "GlorotUniform",
        "henormal": "HeNormal",
        "heuniform": "HeUniform"
    }
    key = KERNEL_INIT if KERNEL_INIT in _KERNEL_INITS else name_map.get(KERNEL_INIT.lower(), KERNEL_INIT)
    if key not in _KERNEL_INITS:
        raise ValueError(f"KERNEL_INIT must be one of {_KERNEL_INITS}, got {KERNEL_INIT}")

    if hidden not in _HIDDEN_CHOICES:
        raise ValueError(f"hidden must be one of {_HIDDEN_CHOICES}, got {hidden}")

    config = {
        "kernel_init": key,
        "hidden": hidden,
        "use_bias": bool(use_bias),
        "output": 10, 
        "LoRA Present":lora_uses,
        "LoRa Rank" : lora_rank,
         }

    if print_use_config:
        print("TensorModels configuration:")
        print(f"  Kernel initializer: {config['kernel_init']}")
        print(f"  Hidden layer size: {config['hidden']}")
        print(f"  Use bias: {config['use_bias']}")
        print(f"  Output size: {config['output']}")
    if (lora_uses == False):
        model = PytorchTensorDigitMist(hidden=hidden, use_bias=use_bias, kernel_init=key)   
    else:
        model = PytorchTensorDigitMistLoRA(hidden=hidden, use_bias=use_bias, kernel_init=key, r=lora_rank)
    return model, config


In [4]:
# =========================
# Instantiate model
# =========================
# --- Model (defined ) ---
torch.manual_seed(42)
model_no_normalizations_nn,cfg = TensorModels(KERNEL_INIT="GlorotUniform", hidden=300)
print(model_no_normalizations_nn)

PytorchTensorDigitMist(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=300, bias=True)
  (fc2): Linear(in_features=300, out_features=150, bias=True)
  (fc3): Linear(in_features=150, out_features=10, bias=True)
)


In [5]:
# Equivalent to Keras model.summary()
summary(model_no_normalizations_nn, input_size=(1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
PytorchTensorDigitMist                   [1, 10]                   --
├─Flatten: 1-1                           [1, 784]                  --
├─Linear: 1-2                            [1, 300]                  235,500
├─Linear: 1-3                            [1, 150]                  45,150
├─Linear: 1-4                            [1, 10]                   1,510
Total params: 282,160
Trainable params: 282,160
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.28
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 1.13
Estimated Total Size (MB): 1.14

In [6]:
# -------------------------
# Compatibility helpers
# -------------------------
def make_grad_scaler_for_device(device: torch.device):
    """Return a GradScaler instance when AMP is appropriate for the device, else None.
    Tries modern API first to avoid FutureWarning, falls back to older APIs."""
    if device.type != "cuda":
        return None
    try:
        return torch.amp.GradScaler("cuda")
    except Exception:
        try:
            return torch.amp.GradScaler()
        except Exception:
            try:
                return torch.cuda.amp.GradScaler()
            except Exception:
                return None

def autocast_ctx(device_type: str):
    """Return an autocast context manager compatible with the installed torch."""
    if hasattr(torch, "autocast"):
        return torch.autocast(device_type=device_type)
    if device_type == "cuda" and hasattr(torch.cuda.amp, "autocast"):
        return torch.cuda.amp.autocast()
    return contextlib.nullcontext()

def _ensure_dataloader(obj, batch_size, shuffle=True, num_workers=2, pin_memory=True):
    """Accept either a DataLoader or a Dataset. Return a DataLoader."""
    if isinstance(obj, torch.utils.data.DataLoader):
        return obj
    if isinstance(obj, torch.utils.data.Dataset):
        return torch.utils.data.DataLoader(obj, batch_size=batch_size, shuffle=shuffle,
                                           num_workers=num_workers, pin_memory=pin_memory)
    raise TypeError("train_loader/test_loader must be a DataLoader or a Dataset")


In [7]:


# -------------------------
# Plot training history
# -------------------------
def plot_history(history: Dict[str, List[float]]) -> None:
    """
    Accepts history with keys:
      - 'train_loss' (list)
      - 'val_loss'   (list or list with None for skipped epochs)
      - 'val_acc'    (list or list with None)
    or the Keras-like keys:
      - 'loss','val_loss','Accuracy','val_Accuracy','epoch'
    The function normalizes both formats and plots loss + accuracy.
    """
    # Normalize keys
    if "train_loss" in history:
        train_loss = history.get("train_loss", [])
        val_loss = history.get("val_loss", [])
        val_acc = history.get("val_acc", [])
        epochs = list(range(1, len(train_loss) + 1))
        acc = None
        # If history contains accuracy under different key names, try to find them
        if "Accuracy" in history:
            acc = history["Accuracy"]
            val_acc = history.get("val_Accuracy", val_acc)
    else:
        # Keras-like
        epochs = history.get("epoch", list(range(1, len(history.get("loss", [])) + 1)))
        train_loss = history.get("loss", [])
        val_loss = history.get("val_loss", [])
        val_acc = history.get("val_Accuracy", history.get("val_acc", []))

    # Plot
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label="train_loss")
    if any(v is not None for v in val_loss):
        plt.plot(epochs, val_loss, label="val_loss")
    plt.title("Loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    if "val_acc" in history or "val_Accuracy" in history:
        # If we have accuracy values
        acc_plot = val_acc if val_acc else history.get("Accuracy", None)
        if acc_plot:
            plt.plot(epochs, acc_plot, label="val_acc")
            plt.title("Validation Accuracy")
            plt.xlabel("epoch")
            plt.ylabel("accuracy")
            plt.grid(True)
            plt.legend()
        else:
            plt.text(0.5, 0.5, "No accuracy data available", ha="center", va="center")
    else:
        plt.text(0.5, 0.5, "No accuracy data available", ha="center", va="center")

    plt.tight_layout()
    plt.show()


# -------------------------
# Predict labels on a loader
# -------------------------
@torch.no_grad()
def predict_labels(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    amp: bool = True
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns (y_true, y_pred) as numpy arrays.
    - model: nn.Module (already moved to device or will be moved here)
    - loader: DataLoader
    - device: torch.device
    - amp: whether to use autocast on CUDA
    """
    model.eval()
    y_true_list = []
    y_pred_list = []

    # choose autocast context safely
    if hasattr(torch, "autocast"):
        autocast_ctx = torch.autocast(device_type=device.type)
    elif device.type == "cuda" and hasattr(torch.cuda.amp, "autocast"):
        autocast_ctx = torch.cuda.amp.autocast()
    else:
        import contextlib
        autocast_ctx = contextlib.nullcontext()

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with autocast_ctx if (amp and device.type == "cuda") else torch.no_grad():
            # If autocast_ctx is a nullcontext, this still works
            probs = model(xb)
        # If model returns logits, take argmax; if it returns log_softmax, argmax still works
        preds = probs.detach().argmax(dim=1).cpu().numpy()
        y_true_list.append(yb.detach().cpu().numpy())
        y_pred_list.append(preds)

    if len(y_true_list) == 0:
        return np.array([]), np.array([])

    y_true = np.concatenate(y_true_list)
    y_pred = np.concatenate(y_pred_list)
    return y_true, y_pred

#-----------------------------------
# upload a single 28×28 image and get the predicted digit
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = MyDigitModel().to(device)
# digit = predict_your_digit(model, "digit.png", device)
#-----------------------------------
    
@torch.no_grad()
def predict_your_digit(
    model: nn.Module,
    image_path: str,
    device: torch.device,
    amp: bool = True
) -> int:
    """
    Predicts a single digit (0–9) from a 28x28 image.
    - model: trained nn.Module
    - image_path: path to image file (28x28 grayscale)
    - device: torch.device
    - amp: whether to use autocast on CUDA
    """

    # Load image
    img = Image.open(image_path).convert("L")  # grayscale
    img = np.array(img, dtype=np.float32) / 255.0  # normalize [0,1]

    # Reshape to (1, 1, 28, 28) for CNNs
    xb = torch.tensor(img).unsqueeze(0).unsqueeze(0).to(device)

    model.eval()

    # choose autocast context safely
    if hasattr(torch, "autocast"):
        autocast_ctx = torch.autocast(device_type=device.type)
    elif device.type == "cuda" and hasattr(torch.cuda.amp, "autocast"):
        autocast_ctx = torch.cuda.amp.autocast()
    else:
        import contextlib
        autocast_ctx = contextlib.nullcontext()

    with autocast_ctx if (amp and device.type == "cuda") else torch.no_grad():
        probs = model(xb)

    pred = probs.detach().argmax(dim=1).item()
    return pred


# -------------------------
# Confusion matrix plot
# -------------------------
def plot_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, labels: List[str] = None, title: str = "Confusion Matrix") -> None:
    """
    Simple confusion matrix heatmap. labels optional list of class names.
    """
    num_classes = int(max(y_true.max(), y_pred.max()) + 1) if y_true.size and y_pred.size else 0
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

##############################
# Plotting for Memory Usage
##############################
def plot_memory_usage(history):
    epochs = range(1, len(history["mem_allocated"]) + 1)
    plt.figure(figsize=(8,5))
    plt.plot(epochs, history["mem_allocated"], label="Allocated (MB)", marker="o")
    plt.plot(epochs, history["mem_reserved"], label="Reserved (MB)", marker="s")
    plt.xlabel("Epoch")
    plt.ylabel("Memory (MB)")
    plt.title("GPU Memory Usage per Epoch")
    plt.legend()
    plt.grid(True)
    plt.show()
    
print("Plotting utilities ready")


Plotting utilities ready


In [8]:
def Ploty_HST(history: Dict[str, list], config: Dict[str, Any], 
              y_true: np.ndarray = None, y_pred: np.ndarray = None) -> None:
    """
    Combined 2x2 plot:
      Row 1: Loss + Validation Accuracy
      Row 2: Confusion Matrix + GPU Memory Usage
    Prints config values and summary stats on top in red bold text.
    """

    # Extract epochs
    epochs = history.get("epoch", list(range(1, len(history.get("train_loss", [])) + 1)))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # -------------------------
    # Top annotation text
    # -------------------------
    kernel_init = config.get("kernel_init", "N/A")
    hidden = config.get("hidden", "N/A")
    lora_present = config.get("LoRA Present", "N/A")
    lora_rank = config.get("LoRa Rank", "N/A")
    total_epoch_time = np.sum(history.get("epoch_time", [])) if history.get("epoch_time") else None
    last_epoch = history["epoch"][-1] if "epoch" in history and history["epoch"] else None
    last_val_loss = history["val_loss"][-1] if history.get("val_loss") else None
    last_val_acc = history["val_acc"][-1] if history.get("val_acc") else None

    # Round to 3 decimals
    last_val_loss_str = f"{last_val_loss:.3f}" if last_val_loss is not None else "None"
    last_val_acc_str  = f"{last_val_acc:.3f}" if last_val_acc is not None else "None"

    summary_text = (
        f"kernel_init={kernel_init}, hidden={hidden}, "
        f"LoRA Present={lora_present}, LoRa Rank={lora_rank}\n"
        f"Total Epoch Time={total_epoch_time:.2f}s, Last Epoch={last_epoch}, "
        f"Last Val Loss={last_val_loss_str}, Last Val Acc={last_val_acc_str}"
    )

    # Red, bold, reduced font
    fig.suptitle(summary_text, fontsize=11, fontweight="bold", color="red")

    # -------------------------
    # Plot 1: Loss
    # -------------------------
    axes[0,0].plot(epochs, history.get("train_loss", []), label="Train Loss")
    if any(v is not None for v in history.get("val_loss", [])):
        axes[0,0].plot(epochs, history.get("val_loss", []), label="Val Loss")
    axes[0,0].set_title("Loss")
    axes[0,0].set_xlabel("Epoch")
    axes[0,0].set_ylabel("Loss")
    axes[0,0].legend()
    axes[0,0].grid(True)

    # -------------------------
    # Plot 2: Validation Accuracy
    # -------------------------
    val_acc = history.get("val_acc", [])
    if any(v is not None for v in val_acc):
        axes[0,1].plot(epochs, val_acc, label="Val Accuracy")
        axes[0,1].set_title("Validation Accuracy")
        axes[0,1].set_xlabel("Epoch")
        axes[0,1].set_ylabel("Accuracy")
        axes[0,1].legend()
        axes[0,1].grid(True)
    else:
        axes[0,1].text(0.5, 0.5, "No accuracy data available", ha="center", va="center")

    # -------------------------
    # Plot 3: Confusion Matrix
    # -------------------------
    if y_true is not None and y_pred is not None and y_true.size and y_pred.size:
        num_classes = int(max(y_true.max(), y_pred.max()) + 1)
        cm = np.zeros((num_classes, num_classes), dtype=np.int64)
        for t, p in zip(y_true, y_pred):
            cm[int(t), int(p)] += 1
        sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", ax=axes[1,0])
        axes[1,0].set_title("Confusion Matrix")
        axes[1,0].set_xlabel("Predicted")
        axes[1,0].set_ylabel("True")
    else:
        axes[1,0].text(0.5, 0.5, "No confusion matrix data", ha="center", va="center")

    # -------------------------
    # Plot 4: GPU Memory Usage
    # -------------------------
    mem_alloc = history.get("mem_allocated", [])
    mem_res   = history.get("mem_reserved", [])
    if mem_alloc and mem_res:
        min_len = min(len(epochs), len(mem_alloc), len(mem_res))
        epochs_plot = epochs[:min_len]
        mem_alloc_plot = mem_alloc[:min_len]
        mem_res_plot   = mem_res[:min_len]

        axes[1,1].plot(epochs_plot, mem_alloc_plot, label="Allocated (MB)", marker="o")
        axes[1,1].plot(epochs_plot, mem_res_plot, label="Reserved (MB)", marker="s")
        axes[1,1].set_title("GPU Memory Usage per Epoch")
        axes[1,1].set_xlabel("Epoch")
        axes[1,1].set_ylabel("Memory (MB)")
        axes[1,1].legend()
        axes[1,1].grid(True)
    else:
        axes[1,1].text(0.5, 0.5, "No memory data available", ha="center", va="center")

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    return fig


In [9]:
# -------------------------
# Early stopper utility
# -------------------------
class EarlyStopper:
    """
    Early stopping helper that supports:
      - stopping when val_acc >= val_acc_target
      - stopping when val_loss <= val_loss_target
      - stopping when monitored metric stops improving for `patience` epochs
    Args:
      monitor: "val_loss", "val_acc", or "both"
      patience: epochs to wait for improvement before stopping
      min_delta: minimum change to qualify as improvement
      val_acc_target: absolute accuracy target to stop early (None to disable)
      val_loss_target: absolute loss target to stop early (None to disable)
      mode: "min" for loss, "max" for accuracy (used when monitor is single metric)
      restore_best: if True, restore model to best observed checkpoint on stop
    """
    def __init__(
        self,
        monitor: str = "val_loss",
        patience: int = 5,
        min_delta: float = 1e-4,
        val_acc_target: Optional[float] = 0.98,
        val_loss_target: Optional[float] = 0.09,
        restore_best: bool = True
    ):
        assert monitor in {"val_loss", "val_acc", "both"}
        self.monitor = monitor
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.val_acc_target = None if val_acc_target is None else float(val_acc_target)
        self.val_loss_target = None if val_loss_target is None else float(val_loss_target)
        self.restore_best = bool(restore_best)

        # internal state
        self.best_metric = None
        self.best_epoch = -1
        self.wait = 0
        self.best_state = None

    def _is_better(self, current, best, mode):
        if best is None:
            return True
        if mode == "min":
            return (best - current) > self.min_delta
        else:  # "max"
            return (current - best) > self.min_delta

    def step(self, epoch: int, model: nn.Module, val_loss: float, val_acc: float) -> bool:
        """
        Call at end of epoch. Returns True if training should stop.
        Also saves best model state internally.
        """
        stop_due_to_target = False
        # Check absolute targets first
        if (self.val_acc_target is not None) and (val_acc >= self.val_acc_target):
            stop_due_to_target = True
        if (self.val_loss_target is not None) and (val_loss <= self.val_loss_target):
            stop_due_to_target = True

        # Determine monitored metric improvement
        # If monitor == both, we use val_loss as primary (min) and val_acc as secondary (max)
        improved = False
        if self.monitor == "val_loss":
            if self._is_better(val_loss, self.best_metric, mode="min"):
                improved = True
                self.best_metric = val_loss
        elif self.monitor == "val_acc":
            if self._is_better(val_acc, self.best_metric, mode="max"):
                improved = True
                self.best_metric = val_acc
        else:  # both
            # require improvement in either metric to count as improvement
            if (self._is_better(val_loss, self.best_metric if isinstance(self.best_metric, float) else None, mode="min")
                or self._is_better(val_acc, self.best_metric if isinstance(self.best_metric, float) else None, mode="max")):
                improved = True
                # store a tuple for bookkeeping (prefer loss for best_metric)
                self.best_metric = val_loss

        if improved:
            self.wait = 0
            self.best_epoch = epoch
            # save best state dict (deep copy)
            try:
                self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            except Exception:
                # fallback: store state_dict reference (less safe)
                self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.wait += 1

        # Stop if absolute target reached OR patience exceeded
        if stop_due_to_target:
            return True
        if self.wait > self.patience:
            return True
        return False

    def restore_best_model(self, model: nn.Module, device: Optional[torch.device] = None):
        """Restore the saved best state into model (if available)."""
        if self.best_state is None:
            return model
        # move tensors back to device and load
        device = device or (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
        state = {k: v.to(device) for k, v in self.best_state.items()}
        model.load_state_dict(state)
        return model



In [10]:
def run_experiment(
    uri=None,
    experiment_name="Digit Mist Pytorch Cuda",
    model_name="default_models",
    run_name="digit_mist_pytorch_cuda",
    selected_models=None,
    monitor="val_loss",
    patience=5,
    min_delta=1e-4,
    val_acc_target=0.98,
    val_loss_target=0.0,
    restore_best=True,
    kernel_init="GlorotUniform",
    hidden=1000,
    lora_uses=True,
    lora_rank=64,
    batch_size_=128,
    epochs_=30,
    validation_freq_=1,
    grad_disable=False,
    tags=None,
):
    """Run a Gradio-configured PyTorch experiment and return log, plot, and summary.""" 

    # MLflow's legacy filesystem tracking backend (for example, ./mlruns) is
    # no longer enabled by default in recent MLflow versions. Use a local
    # SQLite database for experiment tracking and Model Registry metadata.
    # The default database is created in the notebook's current working directory.
    if uri is None or uri == "default_path":
        db_path = (Path.cwd() / "mlflow.db").resolve()
        uri = f"sqlite:///{db_path.as_posix()}"
    mlflow.set_tracking_uri(uri)

    selected_models = list(selected_models or ["Digit Mist Pytorch"])
    unsupported = [m for m in selected_models if m in UNSUPPORTED_MODELS]
    if unsupported:
        return (
            "Training was not started. Unsupported model(s): " + ", ".join(unsupported),
            None,
            "Training failed: unsupported model selected.",
        )

    mlflow.set_experiment(experiment_name)

    early_params = {
        "monitor": monitor or "val_loss",
        "patience": int(patience) if patience is not None else 5,
        "min_delta": float(min_delta) if min_delta is not None else 1e-4,
        "val_acc_target": float(val_acc_target) if val_acc_target is not None else 0.98,
        "val_loss_target": float(val_loss_target) if val_loss_target is not None else 0.0,
        "restore_best": bool(restore_best),
    }
    # All left-column Gradio controls are logged as parameters.
    input_params = {
        "selected_models": ", ".join(selected_models),
        **early_params,
        "KERNEL_INIT": kernel_init or "GlorotUniform",
        "hidden": int(hidden) if hidden is not None else 1000,
        "lora_uses": bool(lora_uses),
        "lora_rank": int(lora_rank) if lora_rank is not None else 64,
        "batch_size_": int(batch_size_) if batch_size_ is not None else 128,
        "epochs_": int(epochs_) if epochs_ is not None else 30,
        "validation_freq_": int(validation_freq_) if validation_freq_ is not None else 1,
        "grad_disable": bool(grad_disable),
    }

    model, cfg = TensorModels(
        KERNEL_INIT=input_params["KERNEL_INIT"],
        hidden=input_params["hidden"],
        print_use_config=True,
        lora_uses=input_params["lora_uses"],
        lora_rank=input_params["lora_rank"],
    )

    with mlflow.start_run(run_name=run_name) as run:
        if tags:
            mlflow.set_tags(tags)
        mlflow.log_params(input_params)

        progress = ProgressList()
        progress.append(f"Selected models: {', '.join(selected_models)}")
        progress.append(f"Early stop params: {early_params}")
        progress.append(
            f"Training params: batch_size={input_params['batch_size_']}, "
            f"epochs={input_params['epochs_']}, "
            f"validation_freq={input_params['validation_freq_']}, "
            f"grad_disable={input_params['grad_disable']}"
        )

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        trained_model, history, cfg = TrainModels(
            model_or_tuple=(model, cfg),
            train_loader=train_dataset,
            test_loader=test_dataset,
            batch_size_=input_params["batch_size_"],
            epochs_=input_params["epochs_"],
            validation_freq_=input_params["validation_freq_"],
            lr=1e-3,
            device=device,
            weight_decay=0.0,
            print_use_config=True,
            early_stop_params=early_params,
            grad_disable=input_params["grad_disable"],
            progress_callback=progress.append,
        )

        summary_report = create_training_summary(history)
        progress.append("Training complete.")
        y_true, y_pred = predict_labels(
            trained_model, test_loader, device=device, amp=True
        )
        plot_fig = Ploty_HST(history, cfg, y_true, y_pred)

        # Log numeric output values represented in the Gradio Summary output.
        completed_epochs = len(history.get("epoch", []))
        val_losses = [float(v) for v in history.get("val_loss", []) if v is not None]
        val_accs = [float(v) for v in history.get("val_acc", []) if v is not None]
        allocated = [float(v) for v in history.get("mem_allocated", [])]
        reserved = [float(v) for v in history.get("mem_reserved", [])]
        output_metrics = {
            "total_epochs_completed": float(completed_epochs),
            "total_epoch_time_seconds": float(sum(history.get("epoch_time", []))),
        }
        if val_losses:
            output_metrics["final_validation_loss"] = val_losses[-1]
        if val_accs:
            output_metrics["final_validation_accuracy"] = val_accs[-1]
        if allocated:
            output_metrics["max_gpu_memory_allocated_mb"] = max(allocated)
        if reserved:
            output_metrics["max_gpu_memory_reserved_mb"] = max(reserved)
        mlflow.log_metrics(output_metrics)

        # Log the selected models, visible summary, and training log as artifacts.
        with tempfile.TemporaryDirectory() as temp_dir:
            selected_file = Path(temp_dir) / "Models Selected from Gradio UI.txt"
            selected_file.write_text("\n".join(selected_models), encoding="utf-8")
            summary_file = Path(temp_dir) / "Gradio Summary Report.txt"
            summary_file.write_text(summary_report, encoding="utf-8")
            log_file = Path(temp_dir) / "Training Log Status.txt"
            log_file.write_text(progress.get_text(), encoding="utf-8")
            mlflow.log_artifact(str(selected_file))
            mlflow.log_artifact(str(summary_file))
            mlflow.log_artifact(str(log_file))

        # Use pickle serialization to avoid PT2's TensorSpec signature requirement.
        # Keep an MNIST-shaped example input so MLflow can record the model input example.
        model_device = next(trained_model.parameters()).device
        input_example = torch.zeros((1, 1, 28, 28), dtype=torch.float32, device=model_device)
        mlflow.pytorch.log_model(
            pytorch_model=trained_model,
            artifact_path=model_name,
            input_example=input_example,
            serialization_format="pickle",
        )
        run_id = run.info.run_id

    registered_model_name = "Digit Mist Pytorch Cuda"
    mlflow.register_model(
        model_uri=f"runs:/{run_id}/{model_name}",
        name=registered_model_name,
    )
    client = MlflowClient()
    latest_versions = client.get_latest_versions(registered_model_name)
    if not latest_versions:
        summary_report += "\n\nMLflow registration completed, but no model version was returned."
        return progress.get_text(), plot_fig, summary_report

    latest_version = max(latest_versions, key=lambda version: int(version.version)).version
    registered_uri = f"models:/{registered_model_name}/{latest_version}"
    # Load the latest registered PyTorch model to verify the registry URI.
    mlflow.pytorch.load_model(registered_uri)

    summary_report += (
        "\n\nMLFLOW MODEL REGISTRY\n"
        + "=" * 50
        + f"\nRegistered model : {registered_model_name}"
        + f"\nModel version    : {latest_version}"
        + f"\nModel URI        : {registered_uri}"
        + f"\nRun ID           : {run_id}"
    )
    return progress.get_text(), plot_fig, summary_report


In [11]:
def TrainModels(
    model_or_tuple,
    train_loader,
    test_loader,
    batch_size_: int = 128,
    epochs_: int = 30,
    validation_freq_: int = 1,
    lr: float = 1e-3,
    device: torch.device | None = None,
    weight_decay: float = 0.0,
    print_use_config: bool = True,
    early_stop_params: Optional[Dict[str, Any]] = None,
    grad_disable: bool = False,
    progress_callback=None
) -> Tuple[nn.Module, Dict[str, list], Dict[str, Any]]:
    """
    Train wrapper with early stopping support.
    grad_disable: if True, validation runs with gradients enabled (for benchmarking).
                  if False, validation runs with torch.no_grad().
    progress_callback: optional callable(text: str) to receive runtime logs.
    Returns: (trained_model, history, config)
    """    
    # -------------------------
    # Logging helper
    # -------------------------
    def _log(msg):
        msg = str(msg)

        if progress_callback is None:
            print(msg)
        elif callable(progress_callback):
            progress_callback(msg)
        elif hasattr(progress_callback, "append"):
            progress_callback.append(msg)
        else:
            print(msg)
    # Unpack model and config
    if isinstance(model_or_tuple, tuple):
        model = model_or_tuple[0]
        config = model_or_tuple[1] if len(model_or_tuple) > 1 else {}
    else:
        model, config = model_or_tuple, {}

    if print_use_config and config:
        _log("Using model config:")
        for k, v in config.items():
            _log(f"  {k}: {v}")

    device = device or (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
    model.to(device)

    # Prepare dataloaders
    if isinstance(train_loader, torch.utils.data.Dataset):
        train_loader = torch.utils.data.DataLoader(
            train_loader, batch_size=batch_size_, shuffle=True, num_workers=2, pin_memory=True
        )
    if isinstance(test_loader, torch.utils.data.Dataset):
        test_loader = torch.utils.data.DataLoader(
            test_loader, batch_size=batch_size_, shuffle=False, num_workers=2, pin_memory=True
        )

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    scaler = make_grad_scaler_for_device(device)

    # Early stopper
    early_stopper = None
    if early_stop_params is not None:
        early_stopper = EarlyStopper(**early_stop_params)

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "epoch_time": [],
        "mem_allocated": [],
        "mem_reserved": []
    }
    total_start = time.time()

    for epoch in range(epochs_):
        epoch_start = time.time()
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad()
            with autocast_ctx(device.type):
                outputs = model(x)
                loss = loss_fn(outputs, y)

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * x.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        history["train_loss"].append(epoch_train_loss)
        history["epoch"].append(epoch + 1)

        # Validation
        if (epoch + 1) % validation_freq_ == 0:
            model.eval()
            val_loss = 0.0
            val_correct = 0

            grad_ctx = contextlib.nullcontext() if grad_disable else torch.no_grad()
            with grad_ctx:
                with autocast_ctx(device.type):
                    for x_val, y_val in test_loader:
                        x_val = x_val.to(device, non_blocking=True)
                        y_val = y_val.to(device, non_blocking=True)
                        preds = model(x_val)
                        val_loss += loss_fn(preds, y_val).item() * x_val.size(0)
                        val_correct += (preds.argmax(dim=1) == y_val).sum().item()

            val_loss /= len(test_loader.dataset)
            val_acc = val_correct / len(test_loader.dataset)

            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)

            if early_stopper is not None:
                stop = early_stopper.step(epoch, model, val_loss=val_loss, val_acc=val_acc)
                if stop:
                    _log(f"Early stopping triggered at epoch {epoch+1}.")
                    if early_stopper.restore_best:
                        model = early_stopper.restore_best_model(model, device=device)
                        _log(f"Restored best model from epoch {early_stopper.best_epoch+1}.")
                    break
        else:
            history["val_loss"].append(None)
            history["val_acc"].append(None)

        # Epoch timing
        epoch_end = time.time()
        epoch_time = epoch_end - epoch_start
        history["epoch_time"].append(epoch_time)

        # Memory monitoring (guard CUDA calls if not available)
        try:
            mem_alloc = torch.cuda.memory_allocated(device) / (1024**2)
            mem_res = torch.cuda.memory_reserved(device) / (1024**2)
        except Exception:
            mem_alloc = 0.0
            mem_res = 0.0
        history["mem_allocated"].append(mem_alloc)
        history["mem_reserved"].append(mem_res)

        _log(
            f"Epoch {epoch+1}/{epochs_} - Train Loss: {epoch_train_loss:.4f}, "
            f"Val Acc: {history['val_acc'][-1]} | ⏱ {epoch_time:.2f}s | "
            f"Mem Alloc: {mem_alloc:.2f} Mb, Mem Reserved: {mem_res:.2f} Mb"
        )

    total_end = time.time()
    total_time = total_end - total_start
    _log(f"✅ Total training time: {total_time:.2f} sec")

    torch.cuda.empty_cache()
    return model, history, config


In [12]:
def figure_to_pil(fig):
    """Convert a Matplotlib figure into a PIL image."""
    buf = io.BytesIO()

    fig.savefig(
        buf,
        format="png",
        dpi=150,
        bbox_inches="tight"
    )

    buf.seek(0)

    with Image.open(buf) as image:
        img = image.convert("RGB")

    plt.close(fig)

    return img

In [13]:
def create_training_summary(history):
    """Create a readable training summary from the training history."""

    epochs = history.get("epoch", [])
    train_loss = history.get("train_loss", [])
    val_loss = history.get("val_loss", [])
    val_acc = history.get("val_acc", [])
    epoch_time = history.get("epoch_time", [])
    mem_allocated = history.get("mem_allocated", [])
    mem_reserved = history.get("mem_reserved", [])

    total_epochs = len(epochs)
    total_epoch_time = sum(epoch_time)

    # Find the last epoch with validation results
    last_val_loss = None
    last_val_acc = None
    last_val_epoch = None

    for i, (loss, acc) in enumerate(zip(val_loss, val_acc)):
        if loss is not None and acc is not None:
            last_val_loss = loss
            last_val_acc = acc
            last_val_epoch = epochs[i] if i < len(epochs) else i + 1

    # Memory figures are measurements recorded after each epoch.
    # These values are not cumulative totals.
    max_allocated = max(mem_allocated, default=0.0)
    max_reserved = max(mem_reserved, default=0.0)

    # Build the per-epoch history table
    lines = [
        "TRAINING SUMMARY",
        "=" * 50,
        f"Total epochs completed : {total_epochs}",
        f"Total epoch time       : {total_epoch_time:.2f} seconds",
        "",
        "LAST VALIDATION RESULTS",
        "-" * 50,
    ]

    if last_val_epoch is not None:
        lines.extend([
            f"Validation epoch       : {last_val_epoch}",
            f"Validation loss        : {last_val_loss:.4f}",
            f"Validation accuracy    : {last_val_acc:.4%}",
        ])
    else:
        lines.append("No validation results available.")

    lines.extend([
        "",
        "GPU MEMORY (recorded per epoch)",
        "-" * 50,
        f"Maximum recorded allocated : {max_allocated:.2f} MB",
        f"Maximum recorded reserved  : {max_reserved:.2f} MB",
        "",
        "EPOCH HISTORY",
        "-" * 50,
        f"{'Epoch':<8}{'Train Loss':<15}{'Val Loss':<15}{'Val Acc':<12}",
        "-" * 50,
    ])

    for i, epoch in enumerate(epochs):
        tr_loss = train_loss[i] if i < len(train_loss) else None
        vl_loss = val_loss[i] if i < len(val_loss) else None
        vl_acc = val_acc[i] if i < len(val_acc) else None

        tr_text = f"{tr_loss:.4f}" if tr_loss is not None else "N/A"
        vl_text = f"{vl_loss:.4f}" if vl_loss is not None else "N/A"
        acc_text = f"{vl_acc:.2%}" if vl_acc is not None else "N/A"

        lines.append(
            f"{epoch:<8}{tr_text:<15}{vl_text:<15}{acc_text:<12}"
        )

    return "\n".join(lines)

In [ ]:
# ---------------------------
# UI helper functions
# ---------------------------
class ProgressList:
    def __init__(self):
        self.lst = []
    def append(self, text):
        try:
            self.lst.append(str(text))
        except Exception:
            self.lst.append(repr(text))
    def clear(self):
        self.lst.clear()
    def get_text(self):
        return "\n".join(self.lst)

UNSUPPORTED_MODELS = {"Digit Mist Neural", "Fashion Mist"}

def on_model_select(selected_models):
    # selected_models is a list or None
    if not selected_models:
        return gr.update(visible=False, value="")
    if any(m in UNSUPPORTED_MODELS for m in selected_models):
        return gr.update(
            value="Models Not Supported: Digit Mist Neural or Fashion Mist selected.",
            visible=True
        )
    return gr.update(visible=False, value="")
    
def run_training(selected_models,
                 monitor, patience, min_delta, val_acc_target, val_loss_target, restore_best,
                 kernel_init, hidden, lora_uses, lora_rank,
                 batch_size_, epochs_, validation_freq_, grad_disable,
                 progress_output, plot_output):
    """
    Called when RUN button is clicked. Builds parameters, calls TensorModels and TrainModels,
    streams logs into progress_output, and returns plot image.
    """
    # Check unsupported models first
    if any(m in UNSUPPORTED_MODELS for m in (selected_models or [])):
        progress_output.append("ERROR: Selected model(s) not supported. Please choose supported models.")
        return None, "ERROR: Unsupported model selected.", \
       "Training was not started because an unsupported model was selected."
    
    # Build early_params with defaults kept if user didn't change
    early_params = {
        "monitor": monitor or "val_loss",
        "patience": int(patience) if patience is not None else 5,
        "min_delta": float(min_delta) if min_delta is not None else 1e-4,
        "val_acc_target": float(val_acc_target) if val_acc_target is not None else 0.98,
        "val_loss_target": float(val_loss_target) if val_loss_target is not None else 0.0,
        "restore_best": bool(restore_best)
    }

    # Build model with user-updated params
    model, cfg = TensorModels(
        KERNEL_INIT=kernel_init or "GlorotUniform",
        hidden=int(hidden) if hidden is not None else 1000,
        print_use_config=True,
        lora_uses=bool(lora_uses),
        lora_rank=int(lora_rank) if lora_rank is not None else 64
    )

    # Clear progress output and stream logs
    progress_output.clear()
    progress_output.append(f"Starting training with model: {model}")
    progress_output.append(f"Early stop params: {early_params}")
    progress_output.append(f"Training params: batch_size={batch_size_}, epochs={epochs_}, validation_freq={validation_freq_}, grad_disable={grad_disable}")

    # Define a simple callback to append logs
    def progress_cb(text):
        progress_output.append(text)

    # Call TrainModels (placeholder)
    trained_model, history, cfg = TrainModels(
        model_or_tuple=(model, cfg),
        train_loader=train_dataset,
        test_loader=test_dataset,
        batch_size_=int(batch_size_),
        epochs_=int(epochs_),
        validation_freq_=int(validation_freq_),
        lr=1e-3,
        device=None,
        weight_decay=0.0,
        print_use_config=True,
        early_stop_params=early_params,
        grad_disable=bool(grad_disable),
        progress_callback=progress_cb
    )

    progress_output.append("Training finished. Running prediction and plotting...")

    # Predict and plot (placeholders)
    device = next(trained_model.parameters()).device
    y_true, y_pred = predict_labels(
        trained_model,
        test_loader,
        device=device,
        amp=True
      )
    plot_fig = Ploty_HST(history, cfg, y_true, y_pred)
    # Generate the training summary
    summary_text = create_training_summary(history)
    status = "Training complete."
    return plot_fig, status, summary_text



# ---------------------------
# Build Gradio UI
# ---------------------------
with gr.Blocks(title="Run Models") as demo:
    gr.Markdown("## Run Models")
    with gr.Row():
        with gr.Column(scale=1):
            # Left column: inputs
            gr.Markdown("### Inputs")

            # 1) Select Models (CheckboxGroup)
            model_select = gr.CheckboxGroup(
                choices=["Digit Mist Pytorch", "Digit Mist Neural", "Fashion Mist"],
                label="Select Models",
                info="Choose one or more models"
            )
            # Warning box (hidden by default)
            model_warning = gr.Textbox(value="", visible=False, interactive=False, label="Warning", elem_id="model-warning")
            model_select.change(fn=on_model_select, inputs=[model_select], outputs=[model_warning])

            # 3) Set Early Stop Parameter (container)
            gr.Markdown("#### Set Early Stop Parameter")
            monitor = gr.Dropdown(choices=["val_loss", "val_acc"], value="val_loss", label="monitor")
            patience = gr.Slider(minimum=1, maximum=10, step=1, value=5, label="patience")
            min_delta = gr.Number(value=1e-4, label="min_delta")
            val_acc_target = gr.Number(value=0.98, label="val_acc_target")
            val_loss_target = gr.Number(value=0.0, label="val_loss_target")
            restore_best = gr.Checkbox(value=True, label="restore_best")

            # 4) Tensor Models parameter
            gr.Markdown("#### Tensor Models Parameter")
            kernel_init = gr.Dropdown(choices=["GlorotNormal","GlorotUniform","HeNormal","HeUniform"], value="GlorotUniform", label="KERNEL_INIT")
            hidden = gr.Dropdown(choices=[1000,750,500,300,150,75,25], value=1000, label="hidden")
            lora_uses = gr.Checkbox(value=True, label="lora_uses")
            lora_rank = gr.Slider(minimum=2, maximum=128, step=2, value=64, label="lora_rank")

            # 5) Tanning parameter (training)
            gr.Markdown("#### Tanning Parameter")
            batch_size_ = gr.Dropdown(choices=[32,64,128,256], value=128, label="batch_size_")
            epochs_ = gr.Dropdown(choices=[10,20,30,40,50], value=30, label="epochs_")
            validation_freq_ = gr.Dropdown(choices=[1,2,3,4,5], value=1, label="validation_freq_")
            grad_disable = gr.Checkbox(value=False, label="grad_disable")

            # 6) RUN button
            run_btn = gr.Button("RUN", variant="primary")

        with gr.Column(scale=1):
            # Right column: outputs
            gr.Markdown("### Outputs")
        
            training_log = gr.Textbox(
                label="Training Log / Status",
                interactive=False,
                lines=12
            )
        
            # New Summary box
            training_summary = gr.Textbox(
                label="Summary",
                interactive=False,
                lines=18,
                max_lines=30
            )
        
            # Training plots
            plot_image = gr.Image(
                label="Training Plots",
                type="pil"
            )

    # Hook up RUN button
    def run_and_stream(
        selected_models,
        monitor, patience, min_delta, val_acc_target, val_loss_target, restore_best,
        kernel_init, hidden, lora_uses, lora_rank,
        batch_size_, epochs_, validation_freq_, grad_disable
    ):
        try:
            log_text, plot_buf, summary_text = run_experiment(
                selected_models=selected_models,
                monitor=monitor,
                patience=patience,
                min_delta=min_delta,
                val_acc_target=val_acc_target,
                val_loss_target=val_loss_target,
                restore_best=restore_best,
                kernel_init=kernel_init,
                hidden=hidden,
                lora_uses=lora_uses,
                lora_rank=lora_rank,
                batch_size_=batch_size_,
                epochs_=epochs_,
                validation_freq_=validation_freq_,
                grad_disable=grad_disable,
            )
            if isinstance(plot_buf, plt.Figure):
                img = figure_to_pil(plot_buf)
            elif isinstance(plot_buf, Image.Image):
                img = plot_buf
            elif isinstance(plot_buf, np.ndarray):
                img = plot_buf
            else:
                img = None
            return log_text, img, summary_text
        except Exception as e:
            tb = traceback.format_exc()
            err_text = f"ERROR during MLflow experiment:\\n{str(e)}\\n\\n{tb}"
            return err_text, None, "Summary unavailable because training or MLflow logging failed."

    run_btn.click(
        fn=run_and_stream,
        inputs=[
            model_select,
            monitor, patience, min_delta,
            val_acc_target, val_loss_target, restore_best,
            kernel_init, hidden, lora_uses, lora_rank,
            batch_size_, epochs_, validation_freq_, grad_disable
        ],
        outputs=[
            training_log,
            plot_image,
            training_summary
        ]
    )

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


TensorModels configuration:
  Kernel initializer: GlorotUniform
  Hidden layer size: 1000
  Use bias: True
  Output size: 10


2026/10/14 14:59:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/14 14:59:31 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
Registered model 'Digit Mist Pytorch Cuda' already exists. Creating a new version of this model...
2026/10/14 14:59:34 WARNING mlflow.tracking._model_registry.fluent: Run with id a4857e4357444fdcbf6d4bf098b12828 has no artifacts at artifact path 'default_models', registering model based on models:/m-8cd37978aa4e483da248f3feac3482ea instead
Created version '2' of model 'Digit Mist Pytorch Cuda'.
/tmp/ipykernel_11485/2346930225.py:164: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_ve

: 